# NYC Crime Data Machine Learning & Crime Forecasting
The purpose of this notebook is to use predictive models to forecast crime rates and locations by using the patterns identified in the EDA.

In [10]:
#Library imports
import pandas as pd
import numpy as np
import holidays

In [11]:
#load and filter data for only Brooklyn
df = pd.read_csv('nyc_crime_data_cleaned.csv')
bk_df = df[df['BORO_NM'] == 'BROOKLYN'].copy()
bk_df['CMPLNT_FR_DTTM'] = pd.to_datetime(bk_df['CMPLNT_FR_DTTM'])
bk_df['date'] = bk_df['CMPLNT_FR_DTTM'].dt.date
print(f"Total Brooklyn crimes: {len(bk_df)}")

Total Brooklyn crimes: 14791


C:\Users\Nicol\AppData\Local\Temp\ipykernel_14556\2963090608.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('nyc_crime_data_cleaned.csv')


## Time Series Data Formatting
Formatting the data by identifying a proper time unit of analysis and creating a target variable

In [12]:
#Create 4-hour time windows/shifts where the bins are as follows: 0-4, 4-8, 8-12, 12-16, 16-20, 20-24
bk_df['hour'] = bk_df['CMPLNT_FR_DTTM'].dt.hour
bk_df['shift_window'] = (bk_df['hour']//4)*4 

#Count crimes per precinct per shift (target variable 'y' which is incident count)
crime_counts = bk_df.groupby(['date', 'shift_window', 'ADDR_PCT_CD']).size().reset_index(name='incident_count')

#Create the overall time grid for ML dataset which ensures there are rows for every date, shift, and precinct combination
all_dates = pd.date_range(start=bk_df['date'].min(), end=bk_df['date'].max()).date
all_shifts = [0, 4, 8, 12, 16, 20]
all_precincts = bk_df['ADDR_PCT_CD'].unique()

#Create a multiIndex for the full grid and convert to a dataframe
index = pd.MultiIndex.from_product([all_dates, all_shifts, all_precincts], names=['date', 'shift_window', 'ADDR_PCT_CD'])
master_grid = pd.DataFrame(index=index).reset_index()

#Merge crimes into the overall time grid
ml_data = pd.merge(master_grid, crime_counts, on=['date', 'shift_window', 'ADDR_PCT_CD'], how='left')

#Fill null counts with 0 since no crimes were reported in these windows
ml_data['incident_count'] = ml_data['incident_count'].fillna(0).astype(int)

#Preview the structured data
print(f"Total rows for ML: {len(ml_data)}")
print(ml_data.head())

Total rows for ML: 113880
         date  shift_window  ADDR_PCT_CD  incident_count
0  2017-01-02             0           75               0
1  2017-01-02             0           70               0
2  2017-01-02             0           66               0
3  2017-01-02             0           83               1
4  2017-01-02             0           62               0


## Feature Engineering
Incorporating cyclical time encoding, lag features, binary flags, and spatial interactions to incorporate seasonality trends into the data since time of the year does affect crime rates

##### Cyclical Time Encoding and Binary Flags

In [13]:
#Convert to datetime for feature engineering
ml_data['date'] = pd.to_datetime(ml_data['date'])

#Cyclical time (shift window) 
ml_data['shift_sin'] = np.sin(2 * np.pi * ml_data['shift_window'] / 24)
ml_data['shift_cos'] = np.cos(2 * np.pi * ml_data['shift_window'] / 24)

#Setup holidays
us_holidays = holidays.US(years=[2017, 2018, 2019])

#For each date, check if it's a holiday and create a binary feature
ml_data['is_holiday'] = ml_data['date'].dt.date.apply(lambda x: x in us_holidays).astype(int)

#Create other time-based features
ml_data['day_of_week'] = ml_data['date'].dt.dayofweek
ml_data['is_weekend'] = ml_data['day_of_week'].isin([5, 6]).astype(int)
ml_data['month'] = ml_data['date'].dt.month

##### Temporal Lag Features
These lag features allow the model to see the "short-term history" of a precinct.

In [ ]:
#sort by precinct and time to ensure lags are chronological
ml_data = ml_data.sort_values(['ADDR_PCT_CD', 'date', 'shift_window'])

#lag feature 1: crime count is in the previous 4-hour window
ml_data['lag_4h'] = ml_data.groupby('ADDR_PCT_CD')['incident_count'].shift(1)

#lag feature 2: crime count is exactly 24 hours ago
ml_data['lag_24h'] = ml_data.groupby('ADDR_PCT_CD')['incident_count'].shift(6)

#first rows of each precinct won't have lags
ml_data = ml_data.fillna(0)

#Calculate rolling statistics
ml_data = ml_data.sort_values(['ADDR_PCT_CD', 'date', 'shift_window'])

#7 day rolling average
ml_data['rolling_mean_7d'] = ml_data.groupby('ADDR_PCT_CD')['incident_count'].transform(
    lambda x: x.shift(1).rolling(window=42, min_periods=1).mean()
)

ml_data['rolling_std_7d'] = ml_data.groupby('ADDR_PCT_CD')['incident_count'].transform(
    lambda x: x.shift(1).rolling(window=42, min_periods=1).std()
)

ml_data[['rolling_mean_7d', 'rolling_std_7d']] = ml_data[['rolling_mean_7d', 'rolling_std_7d']].fillna(0)

##### Spatial Interaction
This identifies a precinct's patterns of behaviour in crime reports (e.g., street-heavy vs. residential-heavy crimes), allowing the model to understand that different environments react differently to the same event.

In [ ]:
#First spatial flag group (housing and street density)
precinct_profile = df.groupby('ADDR_PCT_CD')['PREM_TYP_DESC'].value_counts(normalize=True).unstack(fill_value=0)

#Focus on major EDA findings
precinct_profile = precinct_profile[['STREET', 'RESIDENCE - PUBLIC HOUSING']].copy()

#Binary flags based on medians
precinct_profile['is_street_heavy'] = (precinct_profile['STREET'] > precinct_profile['STREET'].median()).astype(int)
precinct_profile['is_housing_heavy'] = (precinct_profile['RESIDENCE - PUBLIC HOUSING'] > precinct_profile['RESIDENCE - PUBLIC HOUSING'].median()).astype(int)

#Second spatial flag group (commercial zones)
retail_types = ['CHAIN STORE', 'DEPARTMENT STORE', 'GROCERY/BODEGA']
df['is_retail'] = df['PREM_TYP_DESC'].isin(retail_types)

#Calculate retail density per precinct
commercial_profile = df.groupby('ADDR_PCT_CD')['is_retail'].mean().reset_index()
commercial_profile.columns = ['ADDR_PCT_CD', 'retail_density']

#Identify commercial zones
commercial_profile['is_commercial_zone'] = (commercial_profile['retail_density'] > commercial_profile['retail_density'].median()).astype(int)

#Merge into ML dataset
ml_data = ml_data.merge(
    precinct_profile[['is_street_heavy', 'is_housing_heavy']], 
    on='ADDR_PCT_CD', 
    how='left'
)
ml_data = ml_data.merge(
    commercial_profile[['ADDR_PCT_CD', 'is_commercial_zone']], 
    on='ADDR_PCT_CD', 
    how='left'
).fillna(0)